In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BIGRU_BASELINE/glove.6B.zip"
extract_path = "/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction finished")

Extraction finished


# **1**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/1.RAW.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/1.RAW.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/1549339446.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 13660/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:20<16:33, 20.27s/it]

LR: 0.000203
Ep1 | TL 0.554 VL 0.450 | TA 0.707 VA 0.788
LR: 0.000203
Ep2 | TL 0.459 VL 0.405 | TA 0.782 VA 0.812


  4%|▍         | 2/50 [00:39<15:51, 19.82s/it]

...

 32%|███▏      | 16/50 [05:15<11:12, 19.77s/it]

LR: 0.0001015
Ep16 | TL 0.276 VL 0.382 | TA 0.881 VA 0.844


 32%|███▏      | 16/50 [05:35<11:52, 20.95s/it]

LR: 0.0001015
Ep17 | TL 0.248 VL 0.368 | TA 0.895 VA 0.847
Early stop



TEST RESULTS
Acc=0.853 Prec=0.848 Rec=0.860 F1=0.854

EFFICIENCY REPORT:
Avg Time/Epoch: 19.54s
Inference Latency: 0.1481 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/1.RAW.20260309_151712/timing_statistics.json


# **2**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/2.L{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/2.L.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/1971498932.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 16037/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:19<15:50, 19.39s/it]

LR: 0.000203
Ep1 | TL 0.560 VL 0.440 | TA 0.706 VA 0.801


  4%|▍         | 2/50 [00:38<15:28, 19.35s/it]

...

 42%|████▏     | 21/50 [06:53<09:30, 19.66s/it]

LR: 0.0001015
Ep21 | TL 0.212 VL 0.349 | TA 0.912 VA 0.865


 42%|████▏     | 21/50 [07:13<09:58, 20.64s/it]

LR: 0.0001015
Ep22 | TL 0.183 VL 0.375 | TA 0.926 VA 0.866
Early stop



TEST RESULTS
Acc=0.859 Prec=0.843 Rec=0.883 F1=0.863

EFFICIENCY REPORT:
Avg Time/Epoch: 19.53s
Inference Latency: 0.1552 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/2.L20260309_152317/timing_statistics.json


# **3**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/3.R.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/3.R.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/2043490015.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 18402/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:20<16:44, 20.50s/it]

LR: 0.000203
Ep1 | TL 0.498 VL 0.374 | TA 0.753 VA 0.833
LR: 0.000203
Ep2 | TL 0.394 VL 0.350 | TA 0.822 VA 0.843


  6%|▌         | 3/50 [01:01<15:59, 20.40s/it]

...

 40%|████      | 20/50 [06:50<10:11, 20.39s/it]

LR: 5.075e-05
Ep20 | TL 0.208 VL 0.333 | TA 0.915 VA 0.869


 40%|████      | 20/50 [07:10<10:46, 21.54s/it]

LR: 5.075e-05
Ep21 | TL 0.193 VL 0.317 | TA 0.921 VA 0.876
Early stop



TEST RESULTS
Acc=0.868 Prec=0.877 Rec=0.856 F1=0.867

EFFICIENCY REPORT:
Avg Time/Epoch: 20.36s
Inference Latency: 0.1578 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/3.R.20260309_153041/timing_statistics.json


# **4**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/4.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/4.S.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/2182551574.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 18413/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:20<16:20, 20.01s/it]

LR: 0.000203
Ep1 | TL 0.601 VL 0.501 | TA 0.670 VA 0.752
LR: 0.000203
Ep2 | TL 0.501 VL 0.438 | TA 0.754 VA 0.791


  4%|▍         | 2/50 [00:40<16:13, 20.28s/it]

...

 56%|█████▌    | 28/50 [09:39<07:44, 21.11s/it]

LR: 5.075e-05
Ep28 | TL 0.161 VL 0.476 | TA 0.937 VA 0.840


 56%|█████▌    | 28/50 [10:00<07:51, 21.45s/it]

LR: 5.075e-05
Ep29 | TL 0.147 VL 0.477 | TA 0.943 VA 0.844
Early stop



TEST RESULTS
Acc=0.846 Prec=0.852 Rec=0.838 F1=0.845

EFFICIENCY REPORT:
Avg Time/Epoch: 20.55s
Inference Latency: 0.1580 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/4.S.20260309_153802/timing_statistics.json


# **5**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/5.R.L{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/5.R.L.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/1371505496.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 16143/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:22<18:14, 22.34s/it]

LR: 0.000203
Ep1 | TL 0.504 VL 0.378 | TA 0.750 VA 0.832
LR: 0.000203
Ep2 | TL 0.403 VL 0.348 | TA 0.817 VA 0.848


  4%|▍         | 2/50 [00:44<17:52, 22.35s/it]

...

 24%|██▍       | 12/50 [04:26<14:01, 22.16s/it]

LR: 0.0001015
Ep12 | TL 0.273 VL 0.312 | TA 0.883 VA 0.870


 24%|██▍       | 12/50 [04:48<15:14, 24.06s/it]

LR: 0.0001015
Ep13 | TL 0.254 VL 0.320 | TA 0.893 VA 0.866
Early stop



TEST RESULTS
Acc=0.864 Prec=0.869 Rec=0.856 F1=0.863

EFFICIENCY REPORT:
Avg Time/Epoch: 22.04s
Inference Latency: 0.1604 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/5.R.L20260309_154815/timing_statistics.json


# **6.R.S**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/6.R.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/6.R.S.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/1253053961.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 18410/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:22<18:05, 22.16s/it]

LR: 0.000203
Ep1 | TL 0.564 VL 0.470 | TA 0.701 VA 0.778
LR: 0.000203
Ep2 | TL 0.465 VL 0.410 | TA 0.777 VA 0.811


  4%|▍         | 2/50 [00:44<17:42, 22.13s/it]

...

 54%|█████▍    | 27/50 [10:06<08:35, 22.41s/it]

LR: 5.075e-05
Ep27 | TL 0.149 VL 0.440 | TA 0.941 VA 0.859


 54%|█████▍    | 27/50 [10:27<08:54, 23.25s/it]

LR: 5.075e-05
Ep28 | TL 0.131 VL 0.476 | TA 0.948 VA 0.856
Early stop



TEST RESULTS
Acc=0.853 Prec=0.842 Rec=0.869 F1=0.855

EFFICIENCY REPORT:
Avg Time/Epoch: 22.24s
Inference Latency: 0.1654 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/6.R.S.20260309_155314/timing_statistics.json


# **7**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/7.L.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/7.L.S.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/866632542.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 18323/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:20<17:03, 20.89s/it]

LR: 0.000203
Ep1 | TL 0.603 VL 0.530 | TA 0.670 VA 0.727
LR: 0.000203
Ep2 | TL 0.507 VL 0.447 | TA 0.749 VA 0.785


  4%|▍         | 2/50 [00:41<16:41, 20.87s/it]

...

 36%|███▌      | 18/50 [06:20<11:13, 21.05s/it]

LR: 0.0001015
Ep18 | TL 0.275 VL 0.372 | TA 0.882 VA 0.840


 36%|███▌      | 18/50 [06:41<11:54, 22.32s/it]

LR: 0.0001015
Ep19 | TL 0.246 VL 0.376 | TA 0.895 VA 0.842
Early stop



TEST RESULTS
Acc=0.845 Prec=0.849 Rec=0.838 F1=0.844

EFFICIENCY REPORT:
Avg Time/Epoch: 20.96s
Inference Latency: 0.1552 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/7.L.S.20260309_160352/timing_statistics.json


# **8.R.L.S**

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 6 mar 2026

- split index
- embedding false
- tanpa preprocess lagi. murni model aja
- optimizer adamw aja.

@author: indri
"""

# =========================================================
# REVIEWER-GRADE FULL PIPELINE: BiGRU + GloVe
# =========================================================

import os, json, re, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
CONFIG = {
    "max_len": 128,
    "batch_size": 32,
    "epochs": 50,
    "lr": 2.03e-4,
    "embed_dim": 300,
    "hidden": 256,
    "min_freq": 2,
    "max_vocab": 30000,
    "seed": 42,
    "patience": 4,
    "test_size": 0.2,
    "val_size": 0.1,
    "glove_path": r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/GLOVE/glove.6B.300d.txt"
}

# =========================================================
# SEED + DEVICE
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# TOKENIZER
# =========================================================
"""
def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " <URL> ", t)
    t = re.sub(r"[^a-z0-9<>!?.,']+", " ", t)
    return t.split()
"""
# =========================================================
# TOKENIZER
# =========================================================

def clean_text(t):
    return t.split()
# =========================================================
# VOCAB
# =========================================================

def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(clean_text(t))
    vocab = {"<PAD>":0, "<UNK>":1}
    for w, f in c.most_common(CONFIG["max_vocab"]):
        if f >= CONFIG["min_freq"]:
            vocab[w] = len(vocab)
    return vocab

def encode(text, vocab):
    toks = clean_text(text)
    ids = [vocab.get(w, 1) for w in toks][:CONFIG["max_len"]]
    l = len(ids)
    ids += [0]*(CONFIG["max_len"]-l)
    return ids, l


# =========================================================
# DATASET
# =========================================================

def synonym_replace(tokens, p=0.1):
    new = tokens.copy()
    for i in range(len(new)):
        if random.random() < p:
            new[i] = new[i]  # placeholder kalau belum pakai wordnet
    return new

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p]

def random_swap(tokens, n=1):
    new = tokens.copy()
    for _ in range(n):
        i, j = random.sample(range(len(new)), 2)
        new[i], new[j] = new[j], new[i]
    return new

def augment(tokens):
    r = random.random()
    if r < 0.33:
        return synonym_replace(tokens)
    elif r < 0.66:
        return random_delete(tokens)
    else:
        return random_swap(tokens)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, augment=False):
        self.texts = texts
        self.labels = labels.values
        self.vocab = vocab
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = clean_text(self.texts[idx])

        if self.augment:
            if random.random() < 0.5:
                tokens = augment(tokens)

        ids = [self.vocab.get(w,1) for w in tokens][:CONFIG["max_len"]]
        l = len(ids)
        ids += [0]*(CONFIG["max_len"]-l)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "len": torch.tensor(l, dtype=torch.long),
            "y": torch.tensor(self.labels[idx], dtype=torch.long)
        }
# =========================================================
# GLOVE EMBEDDINGS
# =========================================================
def load_glove_embeddings(vocab):
    embeddings = np.random.normal(scale=0.1, size=(len(vocab), CONFIG["embed_dim"]))
    found = 0
    with open(CONFIG["glove_path"], encoding="utf8") as f:
        for line in f:
            sp=line.split()
            w=sp[0]
            if w in vocab:
                embeddings[vocab[w]]=np.asarray(sp[1:],dtype=np.float32)
                found+=1
    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embeddings, dtype=torch.float32)

# =========================================================
# MODEL
# =========================================================
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embeddings=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, CONFIG["embed_dim"], padding_idx=0)
        if embeddings is not None:
            self.emb.weight.data.copy_(embeddings)
            self.emb.weight.requires_grad = False

        self.gru = nn.GRU(
            CONFIG["embed_dim"],
            CONFIG["hidden"],
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3 if 2 > 1 else 0
        )
        self.dropout = nn.Dropout(0.5)
        self.emb_dropout = nn.Dropout(0.4)
        self.gru_dropout = 0.3
        self.fc = nn.Linear(CONFIG["hidden"]*2, 2)

    def forward(self, x, lengths):
        x = self.emb_dropout(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h)

# =========================================================
# LOAD DATA
# =========================================================


def main():
    RUN_DIR =rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/8.R.L.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    json.dump(CONFIG, open(f"{RUN_DIR}/config.json","w"), indent=4)

    #OUTPUT_DIR = r"FINAL TESIS\BIGRUBASELINE_0_20260224_141853"
    #DATA_PATH = r"D:\TOPIK RISET\LATIAN\PREPROCESSING\PREPROCESSING_20260215_135559\11.R-L-S.csv"
    #os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/8.R.L.S.csv"
    df = pd.read_csv(DATA_PATH)
    """
    texts = df["review"].astype(str).tolist()
    labels = df["sentiment"].astype(int).tolist()"""

    #===============================================
    # KHUSUS KALAU MAU SAMPLE 10% DATASETS
    #===============================================

    """
    subset = 0.05

    train_idx = train_idx[:int(len(train_idx)*subset)]
    val_idx   = val_idx[:int(len(val_idx)*subset)]
    test_idx  = test_idx[:int(len(test_idx)*subset)]

    """
    #===============================================

    # 1. Ambil data mentah
    all_reviews = df['review'].astype(str).values
    all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

    train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
    val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
    test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")


    # 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
    train_texts = [all_reviews[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    val_texts = [all_reviews[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    test_texts = [all_reviews[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    # 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
    assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

    # 4. Bangun vocab HANYA dari train_texts
    vocab = build_vocab(train_texts)
    embeddings = load_glove_embeddings(vocab)

    # 5. Dataset (Gunakan list langsung, lebih stabil)
    train_dataset = IMDBDataset(train_texts, pd.Series(train_labels), vocab, augment=True)
    val_dataset   = IMDBDataset(val_texts, pd.Series(val_labels), vocab, augment=False)
    test_dataset  = IMDBDataset(test_texts, pd.Series(test_labels), vocab, augment=False)
    g = torch.Generator()
    g.manual_seed(CONFIG["seed"])

    def seed_worker(worker_id):
        worker_seed = (CONFIG["seed"]) + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = DataLoader(train_dataset,
                              batch_size=(CONFIG["batch_size"]),
                              shuffle=True,
                              worker_init_fn=seed_worker,
                              generator=g)

    val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
    test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

    # =========================================================
    # =========================================================
    model = BiGRU(len(vocab), embeddings).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",        # karena monitor val_acc
        factor=0.5,        # lr dikali 0.5 saat stagnan
        patience=2,        # tunggu 1 epoch stagnan
        min_lr=1e-6,
        #verbose=True
    )
    crit = nn.CrossEntropyLoss()

    # =========================================================
    # CHECKPOINT
    # =========================================================
    CKPT=os.path.join(RUN_DIR,"last.pt")
    start_ep=1
    best_acc=-1
    pat=0

    if os.path.exists(CKPT):
        ck=torch.load(CKPT,map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        best_acc=ck["best"]
        start_ep=ck["ep"]+1
        pat=ck["pat"]
        torch.set_rng_state(ck["torch_rng"])
        np.random.set_state(ck["numpy_rng"])
        random.setstate(ck["python_rng"])
        print("Resumed from epoch",start_ep)

    def test_clean_text_independence():
        text = "This is a TEST!"
        result_1 = clean_text(text)

        # Bayangkan kita punya dataset raksasa
        fake_global_stats = [random.random() for _ in range(1000)]

        result_2 = clean_text(text)

        assert result_1 == result_2, "Fungsi clean_text tidak konsisten!"
        print("Clean_text aman: Tidak terpengaruh data eksternal.")

    test_clean_text_independence()

    # =========================================================
    # TRAIN / EVAL FUNCTION
    # =========================================================
    def run(loader, train=True):
        model.train() if train else model.eval()
        preds, ys, probs = [], [], []
        loss_sum = 0
        for b in loader:
            x = b["ids"].long().to(device)
            l = b["len"].to(device)
            y = b["y"].to(device)
            if train: opt.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(x, l)
                loss = crit(out, y)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            loss_sum += loss.item()*y.size(0)
            p = torch.softmax(out,1)[:,1]
            preds += torch.argmax(out,1).cpu().tolist()
            ys += y.cpu().tolist()
            probs += p.detach().cpu().tolist()
        return loss_sum/len(ys), accuracy_score(ys,preds), ys, preds, probs

    print("========== DEBUG ==========")
    print("Device:", device)
    print("Train size:", len(train_dataset))
    print("Val size:", len(val_dataset))
    print("Test size:", len(test_dataset))
    print("Train batches:", len(train_loader))
    print("Vocab size:", len(vocab))
    print("===========================")

    # =========================================================
    # TRAINING LOOP
    # =========================================================

    patience = pat
    log = []
    epoch_times=[]
    start_time = time.time()

    from tqdm import tqdm
    for ep in tqdm(range(start_ep, CONFIG["epochs"]+1)):

        t0=time.time()

        tr_loss, tr_acc, _, _, _ = run(train_loader, True)
        va_loss, va_acc, _, _, _ = run(val_loader, False)
        scheduler.step(va_acc)
        current_lr = opt.param_groups[0]["lr"]
        print("LR:", current_lr)
        epoch_times.append(time.time()-t0)
        log.append([ep, tr_loss, va_loss, tr_acc, va_acc])
        print(f"Ep{ep} | TL {tr_loss:.3f} VL {va_loss:.3f} | TA {tr_acc:.3f} VA {va_acc:.3f}")
        if va_acc > best_acc:
            best_acc = va_acc
            patience = 0
            torch.save(model.state_dict(), f"{RUN_DIR}/best_model.pt")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print("Early stop")
                break

        # save checkpoint
        torch.save({
            "ep":ep,
            "model":model.state_dict(),
            "opt":opt.state_dict(),
            "sched": scheduler.state_dict(),   # ← tambah ini
            "best":best_acc,
            "pat":patience,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": np.random.get_state(),
            "python_rng": random.getstate()
        },CKPT)

    # =========================================================
    # TEST EVAL
    # =========================================================
    model.load_state_dict(torch.load(f"{RUN_DIR}/best_model.pt"))
    test_loss, test_acc, ys, preds, probs = run(test_loader, False)
    prec = precision_score(ys, preds)
    rec = recall_score(ys, preds)
    f1 = f1_score(ys, preds)

    print("\nTEST RESULTS")
    print(f"Acc={test_acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")

    # =========================================================
    # REVISED INFERENCE TIME CALCULATION
    # =========================================================
    model.eval()
    num_samples = len(test_dataset)
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    total_ms = 0

    with torch.no_grad():
        for b in test_loader:
            x = b["ids"].to(device)
            l = b["len"].to(device)

            # SINKRONISASI WAJIB UNTUK GPU
            if device.type == 'cuda':
                torch.cuda.synchronize()

            starter.record()
            _ = model(x, l)
            ender.record()

            if device.type == 'cuda':
                torch.cuda.synchronize()
                total_ms += starter.elapsed_time(ender)
            else:
                # Jika pakai CPU tetap pakai time.time
                pass

    inference_per_sample_ms = total_ms / num_samples
    # =========================================================
    # TIME LOGGING (FINAL REVISED)
    # =========================-================================
    avg_epoch_dist = np.mean(epoch_times)
    total_train_time = time.time() - start_time

    # Simpan dalam satu dictionary terpusat
    timing_stats = {
        "avg_time_per_epoch_sec": float(avg_epoch_dist),
        "total_training_time_sec": float(total_train_time),
        "total_epochs_completed": int(len(epoch_times)),
        "inference_latency_per_sample_ms": float(inference_per_sample_ms),
        "total_test_samples": int(num_samples),
        "device": str(device),
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    # Simpan sebagai JSON
    with open(os.path.join(RUN_DIR, "timing_statistics.json"), "w") as f:
        json.dump(timing_stats, f, indent=4)

    print(f"\nEFFICIENCY REPORT:")
    print(f"Avg Time/Epoch: {avg_epoch_dist:.2f}s")
    print(f"Inference Latency: {inference_per_sample_ms:.4f} ms/sample")
    print(f"Stats saved to: {RUN_DIR}/timing_statistics.json")

    # =========================================================
    # SAVE LOGS
    # =========================================================
    pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])\
      .to_csv(f"{RUN_DIR}/training_log.csv", index=False)

    pd.DataFrame([{
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss
    }]).to_csv(f"{RUN_DIR}/test_metrics.csv", index=False)

    pd.DataFrame(classification_report(ys,preds,output_dict=True))\
    .T.to_csv(os.path.join(RUN_DIR,"classification_report.csv"))

    cm=confusion_matrix(ys,preds)
    pd.DataFrame(cm).to_csv(os.path.join(RUN_DIR,"confusion_matrix.csv"),index=False)

    fpr,tpr,_=roc_curve(ys,probs)
    pd.DataFrame({"fpr":fpr,"tpr":tpr}).to_csv(os.path.join(RUN_DIR,"roc_curve.csv"),index=False)


    # =========================================================
    # ROC + AUC
    # =========================================================
    fpr, tpr, _ = roc_curve(ys, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.legend()
    plt.title("ROC Curve")
    plt.savefig(f"{RUN_DIR}/roc.png")
    plt.close()

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(ys, preds)

    plt.figure()
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center",
                     va="center",
                     color="black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"{RUN_DIR}/cm.png")
    plt.close()

    # =========================================================
    # LEARNING CURVE
    # =========================================================
    df_log = pd.DataFrame(log, columns=["epoch","train_loss","val_loss","train_acc","val_acc"])

    plt.figure()
    plt.plot(df_log["train_acc"], label="Train")
    plt.plot(df_log["val_acc"], label="Val")
    plt.axhline(test_acc, linestyle="--", label="Test")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.savefig(f"{RUN_DIR}/acc_curve.png")
    plt.close()

    plt.figure()
    plt.plot(df_log["train_loss"], label="Train")
    plt.plot(df_log["val_loss"], label="Val")
    plt.axhline(test_loss, linestyle="--", label="Test")
    plt.legend()
    plt.title("Loss Curve")
    plt.savefig(f"{RUN_DIR}/loss_curve.png")
    plt.close()


if __name__ == "__main__":
    main()

<>:66: SyntaxWarning: invalid escape sequence '\S'
<>:66: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1040/1576101272.py:66: SyntaxWarning: invalid escape sequence '\S'
  t = re.sub(r"http\S+", " <URL> ", t)


GloVe: found 18341/30002 words
Clean_text aman: Tidak terpengaruh data eksternal.
========== DEBUG ==========
Device: cuda
Train size: 40000
Val size: 5000
Test size: 5000
Train batches: 1250
Vocab size: 30002


  2%|▏         | 1/50 [00:21<17:57, 21.99s/it]

LR: 0.000203
Ep1 | TL 0.560 VL 0.473 | TA 0.706 VA 0.772
LR: 0.000203
Ep2 | TL 0.464 VL 0.418 | TA 0.778 VA 0.805


  4%|▍         | 2/50 [00:43<17:34, 21.97s/it]

...

 38%|███▊      | 19/50 [06:54<11:08, 21.56s/it]

LR: 5.075e-05
Ep19 | TL 0.247 VL 0.347 | TA 0.895 VA 0.855


 38%|███▊      | 19/50 [07:15<11:50, 22.92s/it]

LR: 5.075e-05
Ep20 | TL 0.232 VL 0.347 | TA 0.903 VA 0.859
Early stop



TEST RESULTS
Acc=0.855 Prec=0.873 Rec=0.831 F1=0.852

EFFICIENCY REPORT:
Avg Time/Epoch: 21.61s
Inference Latency: 0.1597 ms/sample
Stats saved to: /content/drive/MyDrive/Colab Notebooks/TESIS FIX/BiGRU_BASELINE/8.R.L.S.20260309_161046/timing_statistics.json
